In [2]:
# ============================================================
# 09_appendix_package.ipynb
# Appendix / results package builder
#
# Purpose:
#   Collect existing output folders into one clean appendix package.
#
# This file DOES NOT re-estimate models.
# It only copies, indexes, labels, summarizes, and zips outputs
# that already exist from previous notebooks.
#
# Excluded by design:
#   - outputs_authorweight_placebo
#   - outputs_topology_augmented_placebo
#
# Included by design:
#   - author-weight SCM outputs
#   - author-weight TDA diagnostics
#   - baseline SCM outputs
#   - baseline SCM-TDA diagnostics
#   - baseline placebo inference
#   - topology-augmented SCM outputs
#   - topology-augmented TDA check outputs
#   - topology-augmented LOO robustness outputs
#   - main results package from 08_results_tables_figures
# ============================================================

from pathlib import Path
import shutil
import warnings
import pandas as pd

# ------------------------------------------------------------
# 0) Paths
# ------------------------------------------------------------
ROOT = Path(".").resolve()

DIRS = {
    "author": ROOT / "outputs_authorweight",
    "author_tda": ROOT / "outputs_authorweight_tda",
    "baseline": ROOT / "outputs_baseline",
    "baseline_tda": ROOT / "outputs_baseline_tda",
    "baseline_placebo": ROOT / "outputs_baseline_placebo",
    "topology": ROOT / "outputs_topology_augmented",
    "topology_check": ROOT / "outputs_topology_augmented_tda_check",
    "topology_loo": ROOT / "outputs_topology_augmented_loo",
    "main_results": ROOT / "outputs_results_tables_figures",
}

APP_DIR = ROOT / "outputs_appendix"

# ------------------------------------------------------------
# 1) Settings
# ------------------------------------------------------------
CLEAR_EXISTING_APPENDIX = True
MAKE_ZIP = True

KEEP_SUFFIXES = {
    ".csv",
    ".txt",
    ".pdf",
    ".json",
}

# ------------------------------------------------------------
# 2) Appendix section structure
# ------------------------------------------------------------
SECTION_META = {
    "A_authorweight_scm": {
        "title": "Author-Weight SCM Outputs",
        "description": "Author-weight replication outputs, diagnostics, processed data, donor weights, and figures.",
    },
    "B_authorweight_tda": {
        "title": "Author-Weight TDA Diagnostics",
        "description": "Delay embeddings, persistence diagrams, barcodes, and structural summaries using author weights.",
    },
    "C_baseline_scm": {
        "title": "Baseline SCM Outputs",
        "description": "Baseline SCM weights, diagnostics, path data, and replication figures.",
    },
    "D_baseline_tda": {
        "title": "Baseline SCM-TDA Diagnostics",
        "description": "Delay embeddings, persistence diagrams, barcodes, and structural summaries using baseline SCM weights.",
    },
    "E_baseline_placebo": {
        "title": "Baseline SCM Placebo Inference",
        "description": "Baseline placebo distributions, distances, donor weights, rankings, and inference summaries.",
    },
    "F_topology_augmented_comparison": {
        "title": "Topology-Augmented SCM Comparison",
        "description": "Topology-augmented path comparison, lambda-grid outputs, H1 diagnostics, donor weights, and topology-fit summaries.",
    },
    "G_topology_augmented_loo": {
        "title": "Topology-Augmented SCM Leave-One-Out Robustness",
        "description": "Leave-one-out robustness outputs for topology-augmented SCM, including donor-drop paths, weights, retry results, failed runs, centered-gap figures, H1 Wasserstein diagnostics, and post-RMSPE summaries.",
    },
    "H_main_results_package": {
        "title": "Main Results Package from File 08",
        "description": "Main-paper tables, figures, support files, and manifests generated by 08_results_tables_figures.",
    },
    "Z_manifest": {
        "title": "Package Manifest and Index Files",
        "description": "Master manifest, section summaries, README, source-folder status, and package inventory files.",
    },
}

SECTION_DIRS = {
    section: APP_DIR / section
    for section in SECTION_META.keys()
}

# ------------------------------------------------------------
# 3) Clean and create folders
# ------------------------------------------------------------
if CLEAR_EXISTING_APPENDIX and APP_DIR.exists():
    shutil.rmtree(APP_DIR)
    print(f"Removed old appendix package: {APP_DIR}")

for d in [APP_DIR] + list(SECTION_DIRS.values()):
    d.mkdir(parents=True, exist_ok=True)

print("Appendix output directory:", APP_DIR)

# ------------------------------------------------------------
# 4) Helpers
# ------------------------------------------------------------
def is_hidden_or_checkpoint(path: Path) -> bool:
    return any(
        part.startswith(".") or part == ".ipynb_checkpoints"
        for part in path.parts
    )


def list_files_recursive(folder, keep_suffixes=None):
    folder = Path(folder)

    if folder is None or not folder.exists():
        return []

    keep_suffixes = keep_suffixes or KEEP_SUFFIXES
    out = []

    for p in folder.rglob("*"):
        if not p.is_file():
            continue

        if is_hidden_or_checkpoint(p):
            continue

        if p.suffix.lower() not in keep_suffixes:
            continue

        out.append(p.resolve())

    return sorted(out)


def file_type_from_suffix(path):
    suf = Path(path).suffix.lower()

    if suf == ".csv":
        return "csv"
    if suf == ".pdf":
        return "pdf"
    if suf == ".txt":
        return "txt"
    if suf == ".json":
        return "json"

    return suf.replace(".", "")


def safe_copy_unique(src, dst_folder):
    """
    Copy src into dst_folder.
    If filename already exists, create a unique name using parent folder.
    """
    src = Path(src)
    dst_folder = Path(dst_folder)
    dst_folder.mkdir(parents=True, exist_ok=True)

    dst = dst_folder / src.name

    if not dst.exists():
        shutil.copy2(src, dst)
        return dst.resolve()

    parent_tag = src.parent.name
    candidate = dst_folder / f"{src.stem}__{parent_tag}{src.suffix}"

    i = 2
    while candidate.exists():
        candidate = dst_folder / f"{src.stem}__{parent_tag}_{i}{src.suffix}"
        i += 1

    shutil.copy2(src, candidate)
    return candidate.resolve()


def recommended_use_from_name(name):
    n = name.lower()

    if "manifest" in n or "index" in n or "readme" in n:
        return "index"

    if "loo" in n or "drop_top" in n or "retry" in n:
        return "appendix_robustness_check"

    if "main_" in n:
        return "main_or_appendix_reference"

    if "summary" in n:
        return "appendix_table"

    if "histogram" in n:
        return "appendix_figure"

    if "barcode" in n or "diagram" in n:
        return "appendix_tda_diagnostic"

    if "embedding" in n or "point_cloud" in n or "point-cloud" in n:
        return "appendix_tda_diagnostic"

    if "weights" in n:
        return "appendix_weight_table"

    if "paths" in n or "gap" in n:
        return "appendix_path_figure_or_data"

    if "prepost" in n or "metrics" in n:
        return "appendix_metric_table"

    if "lambda" in n or "tradeoff" in n:
        return "appendix_topology_tuning"

    return "appendix_support"


def caption_from_name(name):
    n = name.lower()

    # --------------------------------------------------------
    # Topology-augmented LOO robustness outputs
    # --------------------------------------------------------
    if "topology_augmented_loo_summary" in n:
        return "Leave-one-out robustness summary for topology-augmented SCM."

    if "topology_augmented_loo_decision_summary" in n:
        return "Decision summary for topology-augmented leave-one-out robustness scenarios."

    if "topology_augmented_loo_failed_runs" in n:
        return "Failed or non-converged topology-augmented leave-one-out runs."

    if "topology_augmented_loo_full_sample_paths" in n:
        return "Full-sample path output used as the reference case for topology-augmented LOO."

    if "topology_augmented_loo_full_sample_weights" in n:
        return "Full-sample donor weights used as the reference case for topology-augmented LOO."

    if "centered_gaps" in n and "loo" in n:
        return "Centered-gap comparison figure for topology-augmented leave-one-out robustness."

    if "h1_wasserstein" in n and "loo" in n:
        return "H1 Wasserstein diagnostic figure for topology-augmented leave-one-out robustness."

    if "post_rmspe" in n and "loo" in n:
        return "Post-treatment RMSPE diagnostic figure for topology-augmented leave-one-out robustness."

    if "drop_top" in n and "paths" in n:
        return "Donor-drop synthetic path output for topology-augmented LOO."

    if "drop_top" in n and "weights" in n:
        return "Donor-drop donor-weight output for topology-augmented LOO."

    if "retry" in n and "paths" in n:
        return "Retry synthetic path output for a topology-augmented LOO scenario."

    if "retry" in n and "weights" in n:
        return "Retry donor-weight output for a topology-augmented LOO scenario."

    if "arizona_only_retry_summary" in n:
        return "Arizona-only retry summary for the topology-augmented LOO robustness check."

    # --------------------------------------------------------
    # Main package outputs from 08
    # --------------------------------------------------------
    if "main_pre_post_comparison" in n:
        return "Main comparison table for pre RMSPE, post RMSPE, and post/pre RMSPE ratio."

    if "main_formal_placebo_summary" in n:
        return "Main placebo inference summary table."

    if "main_h1_author_baseline_comparison" in n:
        return "H1 structural mismatch comparison for author-weight and baseline SCM."

    if "main_h1_baseline_vs_topology_comparison" in n:
        return "H1 structural mismatch comparison for baseline and topology-augmented SCM."

    if "main_h1_author_vs_topology_comparison" in n:
        return "H1 structural mismatch comparison for author-weight and topology-augmented SCM."

    if "main_author_vs_topology_paths_and_centered_gaps" in n:
        return "Path and centered-gap comparison for author-weight and topology-augmented SCM."

    # --------------------------------------------------------
    # Figures and SCM outputs
    # --------------------------------------------------------
    if "author_replication_figure13" in n:
        return "Author-weight replication figure."

    if "baseline_replication_figure13" in n:
        return "Baseline SCM replication figure."

    if "baseline_vs_author_paths" in n:
        return "Path comparison between baseline and author-weight synthetic controls."

    if "baseline_vs_topology_augmented_paths" in n:
        return "Path and centered-gap comparison between baseline and topology-augmented SCM."

    if "author_vs_topology_paths" in n:
        return "Path and centered-gap comparison between author-weight and topology-augmented SCM."

    if "standardized_pre_series" in n:
        return "Standardized pre-treatment series plot."

    if "pre_treatment_standardized_path_comparison" in n:
        return "Pre-treatment standardized path comparison."

    if "histogram" in n and "placebo" in n:
        return "Placebo distribution histogram."

    if "tradeoff_scatter" in n:
        return "Trade-off plot for pre-treatment fit and topology mismatch."

    if "lambda_paths" in n:
        return "Metric paths over the lambda grid."

    if "weight_comparison" in n:
        return "Donor-weight comparison plot."

    if "weights" in n:
        return "Synthetic control donor-weight table."

    if "diagnostics" in n:
        return "SCM diagnostic output."

    if "prepost" in n or "metrics" in n:
        return "Pre/post or model-comparison metric table."

    if "selected_lambda" in n:
        return "Selected lambda value for topology-augmented SCM."

    if "paths" in n or "gap" in n:
        return "Synthetic-control path or gap output."

    # --------------------------------------------------------
    # TDA diagnostics
    # --------------------------------------------------------
    if "embedding_summary" in n:
        return "Embedding-level TDA summary table."

    if "h1_diagram" in n:
        return "First-homology persistence diagram data."

    if "h0_diagram" in n:
        return "Zero-homology persistence diagram data."

    if "barcode" in n:
        return "Persistent barcode figure."

    if "persistence_diagram" in n or "persistence_diagrams" in n:
        return "Persistence diagram figure."

    if "embedding" in n:
        return "Delay-embedding point-cloud output."

    return "Appendix support file."


def source_group_from_path(path):
    path = Path(path).resolve()

    for key, folder in DIRS.items():
        try:
            if str(path).startswith(str(folder.resolve())):
                return key
        except Exception:
            pass

    return "unknown"


def add_files_to_section(section_name, files, rows):
    dst_folder = SECTION_DIRS[section_name]
    copied = []

    for src in files:
        src = Path(src)

        if not src.exists():
            continue

        try:
            dst = safe_copy_unique(src, dst_folder)

            rows.append({
                "section": section_name,
                "section_title": SECTION_META[section_name]["title"],
                "file_name": dst.name,
                "original_file_name": src.name,
                "original_path": str(src),
                "copied_path": str(dst),
                "source_group": source_group_from_path(src),
                "file_type": file_type_from_suffix(dst),
                "caption": caption_from_name(dst.name),
                "recommended_use": recommended_use_from_name(dst.name),
            })

            copied.append(dst)

        except Exception as e:
            warnings.warn(f"Could not copy {src}: {e}")

    return copied


def write_section_index(section_name, manifest_df):
    if len(manifest_df) == 0:
        section_df = pd.DataFrame()
    else:
        section_df = manifest_df.loc[
            manifest_df["section"] == section_name
        ].copy()

    if len(section_df) > 0:
        section_df = section_df.sort_values(
            ["file_type", "file_name"]
        ).reset_index(drop=True)

    csv_path = SECTION_DIRS[section_name] / f"{section_name}_index.csv"
    txt_path = SECTION_DIRS[section_name] / f"{section_name}_index.txt"

    section_df.to_csv(csv_path, index=False)

    with open(txt_path, "w", encoding="utf-8") as f:
        if len(section_df) > 0:
            f.write(section_df.to_string(index=False))
        else:
            f.write(f"{section_name}: no files copied.")

    return csv_path, txt_path


# ------------------------------------------------------------
# 5) Check source folders
# ------------------------------------------------------------
source_status_rows = []

for key, folder in DIRS.items():
    exists = folder.exists()
    files = list_files_recursive(folder) if exists else []

    source_status_rows.append({
        "source_group": key,
        "folder": str(folder),
        "exists": bool(exists),
        "n_files_available": int(len(files)),
    })

source_status_df = pd.DataFrame(source_status_rows)

print("\nSource folder status:")
print(source_status_df.to_string(index=False))

# ------------------------------------------------------------
# 6) Collect files by appendix section
# ------------------------------------------------------------
manifest_rows = []

# A. Author-weight SCM outputs
files_A = list_files_recursive(DIRS["author"])
add_files_to_section("A_authorweight_scm", files_A, manifest_rows)

# B. Author-weight TDA diagnostics
files_B = list_files_recursive(DIRS["author_tda"])
add_files_to_section("B_authorweight_tda", files_B, manifest_rows)

# C. Baseline SCM outputs
files_C = list_files_recursive(DIRS["baseline"])
add_files_to_section("C_baseline_scm", files_C, manifest_rows)

# D. Baseline SCM-TDA diagnostics
files_D = list_files_recursive(DIRS["baseline_tda"])
add_files_to_section("D_baseline_tda", files_D, manifest_rows)

# E. Baseline placebo inference
files_E = list_files_recursive(DIRS["baseline_placebo"])
add_files_to_section("E_baseline_placebo", files_E, manifest_rows)

# F. Topology-augmented SCM comparison outputs
files_F = []
files_F += list_files_recursive(DIRS["topology"])
files_F += list_files_recursive(DIRS["topology_check"])
add_files_to_section("F_topology_augmented_comparison", files_F, manifest_rows)

# G. Topology-augmented leave-one-out robustness outputs
files_G = list_files_recursive(DIRS["topology_loo"])
add_files_to_section("G_topology_augmented_loo", files_G, manifest_rows)

# H. Main results package from 08_results_tables_figures
files_H = list_files_recursive(DIRS["main_results"])
add_files_to_section("H_main_results_package", files_H, manifest_rows)

# ------------------------------------------------------------
# 7) Build master manifest
# ------------------------------------------------------------
manifest_df = pd.DataFrame(manifest_rows)

if len(manifest_df) == 0:
    print("\nWarning: no files were copied into appendix package.")
else:
    manifest_df = manifest_df.sort_values(
        ["section", "file_type", "file_name"]
    ).reset_index(drop=True)

    manifest_df["appendix_item_number"] = (
        manifest_df.groupby("section").cumcount() + 1
    )

    manifest_df["appendix_item_id"] = (
        manifest_df["section"].str[0]
        + manifest_df["appendix_item_number"].astype(str).str.zfill(3)
    )

    ordered_cols = [
        "appendix_item_id",
        "section",
        "section_title",
        "file_name",
        "file_type",
        "caption",
        "recommended_use",
        "source_group",
        "original_file_name",
        "original_path",
        "copied_path",
    ]

    manifest_df = manifest_df[
        [c for c in ordered_cols if c in manifest_df.columns]
    ]

manifest_csv = SECTION_DIRS["Z_manifest"] / "appendix_manifest.csv"
manifest_txt = SECTION_DIRS["Z_manifest"] / "appendix_manifest.txt"

manifest_df.to_csv(manifest_csv, index=False)

with open(manifest_txt, "w", encoding="utf-8") as f:
    if len(manifest_df) > 0:
        f.write(manifest_df.to_string(index=False))
    else:
        f.write("No appendix files were copied.")

# ------------------------------------------------------------
# 8) Per-section indexes
# ------------------------------------------------------------
section_index_rows = []

for section_name in SECTION_DIRS:
    if section_name == "Z_manifest":
        continue

    csv_path, txt_path = write_section_index(section_name, manifest_df)

    section_index_rows.append({
        "section": section_name,
        "section_title": SECTION_META[section_name]["title"],
        "index_csv": str(csv_path),
        "index_txt": str(txt_path),
    })

section_index_df = pd.DataFrame(section_index_rows)

section_index_csv = SECTION_DIRS["Z_manifest"] / "appendix_section_indexes.csv"
section_index_txt = SECTION_DIRS["Z_manifest"] / "appendix_section_indexes.txt"

section_index_df.to_csv(section_index_csv, index=False)

with open(section_index_txt, "w", encoding="utf-8") as f:
    if len(section_index_df) > 0:
        f.write(section_index_df.to_string(index=False))
    else:
        f.write("No section indexes available.")

# ------------------------------------------------------------
# 9) Section summary
# ------------------------------------------------------------
summary_rows = []

for section_name in SECTION_META:
    if section_name == "Z_manifest":
        continue

    if len(manifest_df) > 0:
        sub = manifest_df.loc[manifest_df["section"] == section_name]
    else:
        sub = pd.DataFrame()

    summary_rows.append({
        "section": section_name,
        "section_title": SECTION_META[section_name]["title"],
        "n_files": int(len(sub)),
        "n_csv": int((sub["file_type"] == "csv").sum()) if len(sub) else 0,
        "n_pdf": int((sub["file_type"] == "pdf").sum()) if len(sub) else 0,
        "n_txt": int((sub["file_type"] == "txt").sum()) if len(sub) else 0,
        "n_json": int((sub["file_type"] == "json").sum()) if len(sub) else 0,
    })

summary_df = pd.DataFrame(summary_rows)

summary_csv = SECTION_DIRS["Z_manifest"] / "appendix_section_summary.csv"
summary_txt = SECTION_DIRS["Z_manifest"] / "appendix_section_summary.txt"

summary_df.to_csv(summary_csv, index=False)

with open(summary_txt, "w", encoding="utf-8") as f:
    if len(summary_df) > 0:
        f.write(summary_df.to_string(index=False))
    else:
        f.write("No appendix summary available.")

# ------------------------------------------------------------
# 10) Source folder status output
# ------------------------------------------------------------
source_status_csv = SECTION_DIRS["Z_manifest"] / "appendix_source_folder_status.csv"
source_status_txt = SECTION_DIRS["Z_manifest"] / "appendix_source_folder_status.txt"

source_status_df.to_csv(source_status_csv, index=False)

with open(source_status_txt, "w", encoding="utf-8") as f:
    f.write(source_status_df.to_string(index=False))

# ------------------------------------------------------------
# 11) README
# ------------------------------------------------------------
readme_path = SECTION_DIRS["Z_manifest"] / "README_appendix_package.txt"

with open(readme_path, "w", encoding="utf-8") as f:
    f.write(
        "Appendix Results Package\n"
        "========================\n"
        "\n"
        "Generated by: 09_appendix_package.ipynb\n"
        "\n"
        "Purpose\n"
        "-------\n"
        "This package collects existing output files from the synthetic-control,\n"
        "topology-augmented synthetic-control, TDA diagnostic, baseline placebo,\n"
        "and leave-one-out robustness workflows.\n"
        "\n"
        "This notebook does not re-estimate any model. It only copies existing\n"
        "outputs, builds section-level indexes, creates a master manifest, writes\n"
        "summary files, and optionally compresses the package into a zip archive.\n"
        "\n"
        "Excluded Outputs\n"
        "----------------\n"
        "The following folders are intentionally excluded from this package:\n"
        "- outputs_authorweight_placebo\n"
        "- outputs_topology_augmented_placebo\n"
        "\n"
        "Included Source Folders\n"
        "-----------------------\n"
    )

    for key, folder in DIRS.items():
        f.write(f"- {key}: {folder}\n")

    f.write(
        "\n"
        "Package Sections\n"
        "----------------\n"
    )

    for section_name, meta in SECTION_META.items():
        f.write(f"- {section_name}: {meta['title']}\n")
        f.write(f"  {meta['description']}\n")

    f.write(
        "\n"
        "Important Index Files\n"
        "---------------------\n"
        "- Z_manifest/appendix_manifest.csv\n"
        "  Master file-level manifest. Use this first when locating a result.\n"
        "\n"
        "- Z_manifest/appendix_section_summary.csv\n"
        "  Counts of csv, pdf, txt, and json files by section.\n"
        "\n"
        "- Z_manifest/appendix_source_folder_status.csv\n"
        "  Records which source folders existed at package-build time and how many\n"
        "  eligible files were available in each source folder.\n"
        "\n"
        "- Z_manifest/appendix_section_indexes.csv\n"
        "  Points to each section-level index file.\n"
        "\n"
        "- Each non-manifest section contains its own *_index.csv and *_index.txt.\n"
        "\n"
        "Recommended Reading Order\n"
        "-------------------------\n"
        "1. Start with H_main_results_package for the paper-facing tables and figures.\n"
        "2. Use A_authorweight_scm and C_baseline_scm to verify SCM construction.\n"
        "3. Use B_authorweight_tda and D_baseline_tda to inspect topological diagnostics.\n"
        "4. Use E_baseline_placebo for formal baseline placebo inference.\n"
        "5. Use F_topology_augmented_comparison for topology-augmented SCM outputs.\n"
        "6. Use G_topology_augmented_loo for leave-one-out robustness checks.\n"
        "\n"
        "Leave-One-Out Robustness Note\n"
        "-----------------------------\n"
        "The folder G_topology_augmented_loo contains the topology-augmented SCM\n"
        "leave-one-out robustness outputs. It should be treated as a robustness\n"
        "module, not as the main estimation output. Failed or non-converged runs,\n"
        "if present, are intentionally retained and indexed rather than silently\n"
        "removed.\n"
        "\n"
        "Baseline Placebo Note\n"
        "---------------------\n"
        "The package retains baseline placebo inference outputs because they are\n"
        "part of the formal comparison and diagnostic workflow. Author-weight\n"
        "placebo and topology-augmented placebo folders are excluded by design.\n"
        "\n"
        "File Handling Rules\n"
        "-------------------\n"
        "- Hidden folders and .ipynb_checkpoints are ignored.\n"
        "- Only files with these suffixes are copied: "
        f"{sorted(list(KEEP_SUFFIXES))}\n"
        "- File names are preserved unless duplicates require a unique suffix.\n"
        "- If two files have the same name, the copied duplicate receives a suffix\n"
        "  based on its parent folder name.\n"
        "\n"
        "Build Settings\n"
        "--------------\n"
        f"- CLEAR_EXISTING_APPENDIX = {CLEAR_EXISTING_APPENDIX}\n"
        f"- MAKE_ZIP = {MAKE_ZIP}\n"
        f"- KEEP_SUFFIXES = {sorted(list(KEEP_SUFFIXES))}\n"
        "\n"
        "Zip Output\n"
        "----------\n"
        "If MAKE_ZIP is True, the notebook creates:\n"
        "- outputs_appendix_package.zip\n"
        "\n"
        "Interpretation Warning\n"
        "----------------------\n"
        "This package is an output archive. It does not by itself validate the\n"
        "econometric design, causal assumptions, topology choices, or optimizer\n"
        "stability. Those claims must be supported in the paper text using the\n"
        "relevant tables, figures, diagnostics, and robustness results included here.\n"
    )

# ------------------------------------------------------------
# 12) Optional zip package
# ------------------------------------------------------------
zip_path = None

if MAKE_ZIP:
    zip_base = ROOT / "outputs_appendix_package"
    archive_path = shutil.make_archive(
        base_name=str(zip_base),
        format="zip",
        root_dir=str(APP_DIR)
    )
    zip_path = Path(archive_path)
    print("\nCreated zip package:")
    print(zip_path)

# ------------------------------------------------------------
# 13) Final console summary
# ------------------------------------------------------------
print("\n" + "=" * 72)
print("09_appendix_package finished")
print("=" * 72)

print("\nSection summary:")
if len(summary_df) > 0:
    print(summary_df.to_string(index=False))
else:
    print("No files copied.")

print("\nMaster manifest:")
print(manifest_csv)

print("\nSection indexes:")
print(section_index_csv)

print("\nSource folder status:")
print(source_status_csv)

print("\nREADME:")
print(readme_path)

if zip_path is not None:
    print("\nZip package:")
    print(zip_path)

print("\nSection folders:")
for k, v in SECTION_DIRS.items():
    print(f" - {k}: {v}")

print("\nTop-level appendix folder:")
print(APP_DIR)

Removed old appendix package: /home/jupyter/topological fit check on scm/outputs_appendix
Appendix output directory: /home/jupyter/topological fit check on scm/outputs_appendix

Source folder status:
    source_group                                                                          folder  exists  n_files_available
          author                 /home/jupyter/topological fit check on scm/outputs_authorweight    True                  6
      author_tda             /home/jupyter/topological fit check on scm/outputs_authorweight_tda    True                 41
        baseline                     /home/jupyter/topological fit check on scm/outputs_baseline    True                  9
    baseline_tda                 /home/jupyter/topological fit check on scm/outputs_baseline_tda    True                 23
baseline_placebo             /home/jupyter/topological fit check on scm/outputs_baseline_placebo    True                 16
        topology           /home/jupyter/topological fit